Libraries

In [ ]:
import re
import math
from pathlib import Path
from typing import Optional, Dict, Iterable, List, Tuple, Any
import copy
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import nbinom, lognorm, skewnorm, exponweib
from scipy.optimize import minimize
from scipy.stats import (
    poisson,
    nbinom,
    lognorm,
    gamma as gamma_dist,
    weibull_min,
    skewnorm,
    genpareto,
    burr12,
    kstest,
)

Fitting Models to frequency and severity data

In [ ]:
# ============================================================
# CONFIG
# ============================================================
CLEANED_FILE = Path(
    "/Users/ben/Desktop/Uni/ACTL/ACTL4001/Assignment Code/Basic Data Cleaning and Modeling/srcsc-2026-claims-cargo_cleaned.xlsx"
)

FREQ_SHEET = "freq"
SEV_SHEET = "sev"

TEST_SIZE = 0.20
RANDOM_SEED = 42

# Candidate alpha values for NB GLM search
NB_ALPHA_GRID = [0.05, 0.10, 0.20, 0.50, 1.00, 2.00, 5.00]

# Candidate thresholds for spliced severity fit
SPLICE_THRESHOLD_GRID = (0.85, 0.90, 0.95)


# ============================================================
# HELPERS
# ============================================================
def find_first_existing_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for col in candidates:
        if col in df.columns:
            return col
    return None


def normalise_cargo_label(x: str) -> str:
    x = str(x).strip().lower()
    replacements = {
        "rare earth": "rare earths",
        "rare-earths": "rare earths",
        "rare_earths": "rare earths",
    }
    return replacements.get(x, x)


def safe_numeric(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def train_test_split_df(df: pd.DataFrame, test_size: float = 0.20, seed: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.RandomState(seed)
    idx = np.arange(len(df))
    rng.shuffle(idx)

    n_test = int(round(len(df) * test_size))
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]

    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()
    return train_df, test_df


def build_formula(response: str, numeric_cols: List[str], categorical_cols: List[str]) -> str:
    terms = []
    terms.extend(numeric_cols)
    terms.extend(["C({0})".format(c) for c in categorical_cols])

    if len(terms) == 0:
        return "{0} ~ 1".format(response)

    return "{0} ~ {1}".format(response, " + ".join(terms))


def extract_pvalue_table(model) -> pd.DataFrame:
    out = pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "stat": model.tvalues.values if hasattr(model, "tvalues") else np.nan,
        "p_value": model.pvalues.values,
    })

    if hasattr(model, "conf_int"):
        ci = model.conf_int()
        out["ci_low"] = ci.iloc[:, 0].values
        out["ci_high"] = ci.iloc[:, 1].values

    return out.sort_values(["p_value", "term"]).reset_index(drop=True)


def mean_deviance(model, y_true: np.ndarray, mu_pred: np.ndarray) -> float:
    try:
        dev = model.family.deviance(y_true, mu_pred)
        return float(dev) / len(y_true)
    except Exception:
        return np.nan


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def available_predictors(
    df: pd.DataFrame,
    numeric_candidates: List[str],
    categorical_candidates: List[str],
) -> Tuple[List[str], List[str]]:
    numeric_cols = [c for c in numeric_candidates if c in df.columns]
    categorical_cols = [c for c in categorical_candidates if c in df.columns]
    return numeric_cols, categorical_cols


# ============================================================
# LOAD DATA
# ============================================================
def load_clean_data(file_path: Path) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    freq_df = pd.read_excel(file_path, sheet_name=FREQ_SHEET)
    sev_df = pd.read_excel(file_path, sheet_name=SEV_SHEET)

    freq_df = freq_df.loc[:, ~freq_df.columns.duplicated()].copy()
    sev_df = sev_df.loc[:, ~sev_df.columns.duplicated()].copy()

    cargo_candidates = ["cargo_type", "cargo", "cargo_category", "type_of_cargo"]
    freq_candidates = ["claim_count", "claims_count", "n_claims", "claim_num", "claim_frequency"]
    sev_candidates = ["claim_amount", "severity", "claim_size", "loss_amount", "paid_amount"]
    exposure_candidates = ["exposure", "policy_exposure"]

    freq_cargo_col = find_first_existing_column(freq_df, cargo_candidates)
    sev_cargo_col = find_first_existing_column(sev_df, cargo_candidates)
    freq_count_col = find_first_existing_column(freq_df, freq_candidates)
    sev_amount_col = find_first_existing_column(sev_df, sev_candidates)
    exposure_col = find_first_existing_column(freq_df, exposure_candidates)

    if freq_cargo_col is None or sev_cargo_col is None:
        raise ValueError("Could not find cargo_type column in one or both sheets.")
    if freq_count_col is None:
        raise ValueError("Could not find claim_count column in freq sheet.")
    if sev_amount_col is None:
        raise ValueError("Could not find claim_amount column in sev sheet.")

    freq_df[freq_cargo_col] = freq_df[freq_cargo_col].astype(str).map(normalise_cargo_label)
    sev_df[sev_cargo_col] = sev_df[sev_cargo_col].astype(str).map(normalise_cargo_label)

    freq_df = safe_numeric(
        freq_df,
        [freq_count_col, exposure_col, "cargo_value", "weight", "distance", "transit_duration",
         "pilot_experience", "vessel_age", "solar_radiation", "debris_density"]
    )

    sev_df = safe_numeric(
        sev_df,
        [sev_amount_col, "cargo_value", "weight", "distance", "transit_duration",
         "pilot_experience", "vessel_age", "solar_radiation", "debris_density", "claim_seq"]
    )

    # Frequency cleaning
    freq_df = freq_df.dropna(subset=[freq_cargo_col, freq_count_col]).copy()
    freq_df = freq_df[freq_df[freq_count_col] >= 0].copy()
    freq_df = freq_df[np.isclose(freq_df[freq_count_col], np.round(freq_df[freq_count_col]))].copy()
    freq_df[freq_count_col] = freq_df[freq_count_col].astype(int)

    # Severity cleaning
    sev_df = sev_df.dropna(subset=[sev_cargo_col, sev_amount_col]).copy()
    sev_df = sev_df[sev_df[sev_amount_col] > 0].copy()

    meta = {
        "freq_cargo_col": freq_cargo_col,
        "sev_cargo_col": sev_cargo_col,
        "freq_count_col": freq_count_col,
        "sev_amount_col": sev_amount_col,
        "exposure_col": exposure_col,
    }

    return freq_df, sev_df, meta


# ============================================================
# FREQUENCY DISTRIBUTIONS
# ============================================================
def nb2_neg_loglik(params: np.ndarray, y: np.ndarray) -> float:
    log_mu, log_alpha = params
    mu = np.exp(log_mu)
    alpha = np.exp(log_alpha)

    if (not np.isfinite(mu)) or (not np.isfinite(alpha)) or (mu <= 0) or (alpha <= 0):
        return np.inf

    r = 1.0 / alpha
    p = r / (r + mu)

    return -np.sum(nbinom.logpmf(y, n=r, p=p))


def fit_frequency_distributions(freq_df: pd.DataFrame, count_col: str) -> Tuple[pd.DataFrame, pd.Series]:
    y = freq_df[count_col].to_numpy()
    n = len(y)

    rows = []

    # Poisson
    lam = float(np.mean(y))
    ll = float(np.sum(poisson.logpmf(y, mu=lam)))
    rows.append({
        "model": "Poisson",
        "n": n,
        "loglik": ll,
        "aic": 2 * 1 - 2 * ll,
        "bic": math.log(n) * 1 - 2 * ll,
        "mean": lam,
        "alpha": np.nan,
        "size_r": np.nan,
        "pred_zero_pct": 100 * float(poisson.pmf(0, mu=lam)),
        "observed_zero_pct": 100 * float(np.mean(y == 0)),
    })

    # Negative Binomial
    y_mean = float(np.mean(y))
    y_var = float(np.var(y, ddof=1))
    alpha_start = max((y_var - y_mean) / max(y_mean ** 2, 1e-12), 1e-8)

    res = minimize(
        nb2_neg_loglik,
        x0=np.array([np.log(max(y_mean, 1e-8)), np.log(alpha_start)]),
        args=(y,),
        method="L-BFGS-B",
    )

    if res.success:
        mu_nb = float(np.exp(res.x[0]))
        alpha_nb = float(np.exp(res.x[1]))
        r_nb = 1.0 / alpha_nb
        p_nb = r_nb / (r_nb + mu_nb)
        ll_nb = -float(res.fun)

        rows.append({
            "model": "Negative Binomial",
            "n": n,
            "loglik": ll_nb,
            "aic": 2 * 2 - 2 * ll_nb,
            "bic": math.log(n) * 2 - 2 * ll_nb,
            "mean": mu_nb,
            "alpha": alpha_nb,
            "size_r": r_nb,
            "pred_zero_pct": 100 * float(nbinom.pmf(0, n=r_nb, p=p_nb)),
            "observed_zero_pct": 100 * float(np.mean(y == 0)),
        })

    out = pd.DataFrame(rows).sort_values(["aic", "bic"]).reset_index(drop=True)
    best = out.iloc[0]
    return out, best


# ============================================================
# SEVERITY DISTRIBUTIONS
# ============================================================
def fit_lognormal_dist(x: np.ndarray) -> Dict[str, Any]:
    logx = np.log(x)
    mu = float(np.mean(logx))
    sigma = float(np.std(logx, ddof=0))

    if sigma <= 0:
        raise ValueError("sigma <= 0")

    ll = float(np.sum(lognorm.logpdf(x, s=sigma, scale=np.exp(mu))))
    n = len(x)

    def cdf(v):
        return lognorm.cdf(v, s=sigma, scale=np.exp(mu))

    return {
        "model": "lognormal",
        "loglik": ll,
        "aic": 2 * 2 - 2 * ll,
        "bic": math.log(n) * 2 - 2 * ll,
        "ks_stat": float(kstest(x, cdf).statistic),
        "ks_pvalue": float(kstest(x, cdf).pvalue),
        "params": str({"mu_log": mu, "sigma_log": sigma}),
    }


def fit_gamma_distn(x: np.ndarray) -> Dict[str, Any]:
    a, loc, scale = gamma_dist.fit(x, floc=0)
    a = float(a)
    scale = float(scale)
    ll = float(np.sum(gamma_dist.logpdf(x, a=a, loc=0, scale=scale)))
    n = len(x)

    def cdf(v):
        return gamma_dist.cdf(v, a=a, loc=0, scale=scale)

    return {
        "model": "gamma",
        "loglik": ll,
        "aic": 2 * 2 - 2 * ll,
        "bic": math.log(n) * 2 - 2 * ll,
        "ks_stat": float(kstest(x, cdf).statistic),
        "ks_pvalue": float(kstest(x, cdf).pvalue),
        "params": str({"shape": a, "scale": scale}),
    }


def fit_weibull_distn(x: np.ndarray) -> Dict[str, Any]:
    c, loc, scale = weibull_min.fit(x, floc=0)
    c = float(c)
    scale = float(scale)
    ll = float(np.sum(weibull_min.logpdf(x, c=c, loc=0, scale=scale)))
    n = len(x)

    def cdf(v):
        return weibull_min.cdf(v, c=c, loc=0, scale=scale)

    return {
        "model": "weibull_min",
        "loglik": ll,
        "aic": 2 * 2 - 2 * ll,
        "bic": math.log(n) * 2 - 2 * ll,
        "ks_stat": float(kstest(x, cdf).statistic),
        "ks_pvalue": float(kstest(x, cdf).pvalue),
        "params": str({"shape": c, "scale": scale}),
    }


def fit_log_skew_normal_dist(x: np.ndarray) -> Dict[str, Any]:
    logx = np.log(x)
    a, loc, scale = skewnorm.fit(logx)
    a = float(a)
    loc = float(loc)
    scale = float(scale)

    if scale <= 0:
        raise ValueError("scale <= 0")

    ll = float(np.sum(skewnorm.logpdf(logx, a=a, loc=loc, scale=scale) - logx))
    n = len(x)

    def cdf(v):
        v = np.asarray(v)
        out = np.zeros_like(v, dtype=float)
        mask = v > 0
        out[mask] = skewnorm.cdf(np.log(v[mask]), a=a, loc=loc, scale=scale)
        return out

    return {
        "model": "log_skew_normal",
        "loglik": ll,
        "aic": 2 * 3 - 2 * ll,
        "bic": math.log(n) * 3 - 2 * ll,
        "ks_stat": float(kstest(x, cdf).statistic),
        "ks_pvalue": float(kstest(x, cdf).pvalue),
        "params": str({"shape": a, "loc_log": loc, "scale_log": scale}),
    }


def fit_burr12_dist(x: np.ndarray) -> Dict[str, Any]:
    c, d, loc, scale = burr12.fit(x, floc=0)
    c = float(c)
    d = float(d)
    scale = float(scale)

    ll = float(np.sum(burr12.logpdf(x, c=c, d=d, loc=0, scale=scale)))
    n = len(x)

    def cdf(v):
        return burr12.cdf(v, c=c, d=d, loc=0, scale=scale)

    return {
        "model": "burr12",
        "loglik": ll,
        "aic": 2 * 3 - 2 * ll,
        "bic": math.log(n) * 3 - 2 * ll,
        "ks_stat": float(kstest(x, cdf).statistic),
        "ks_pvalue": float(kstest(x, cdf).pvalue),
        "params": str({"c": c, "d": d, "scale": scale}),
    }


def fit_spliced_lognormal_gpd_dist(
    x: np.ndarray,
    threshold_grid: Tuple[float, ...] = (0.85, 0.90, 0.95),
) -> Dict[str, Any]:
    best = None

    for q in threshold_grid:
        u = float(np.quantile(x, q))
        body = x[x <= u]
        tail = x[x > u]

        if len(body) < 30 or len(tail) < 10:
            continue

        log_body = np.log(body)
        mu = float(np.mean(log_body))
        sigma = float(np.std(log_body, ddof=0))

        if sigma <= 0:
            continue

        F_u = float(lognorm.cdf(u, s=sigma, scale=np.exp(mu)))
        if F_u <= 0 or F_u >= 1:
            continue

        excess = tail - u
        xi, loc, beta = genpareto.fit(excess, floc=0)
        xi = float(xi)
        beta = float(beta)

        if beta <= 0:
            continue

        w_body = float(len(body)) / float(len(x))

        ll_body = np.sum(
            np.log(w_body)
            + lognorm.logpdf(body, s=sigma, scale=np.exp(mu))
            - np.log(F_u)
        )

        ll_tail = np.sum(
            np.log(1 - w_body)
            + genpareto.logpdf(excess, c=xi, loc=0, scale=beta)
        )

        ll = float(ll_body + ll_tail)

        def cdf(v, u=u, w_body=w_body, mu=mu, sigma=sigma, F_u=F_u, xi=xi, beta=beta):
            v = np.asarray(v, dtype=float)
            out = np.zeros_like(v)

            mask1 = (v > 0) & (v <= u)
            mask2 = v > u

            out[mask1] = w_body * lognorm.cdf(v[mask1], s=sigma, scale=np.exp(mu)) / F_u
            out[mask2] = w_body + (1 - w_body) * genpareto.cdf(
                v[mask2] - u, c=xi, loc=0, scale=beta
            )
            return out

        ks = kstest(x, cdf)

        candidate = {
            "model": "spliced_lognormal_gpd",
            "loglik": ll,
            "aic": 2 * 5 - 2 * ll,
            "bic": math.log(len(x)) * 5 - 2 * ll,
            "ks_stat": float(ks.statistic),
            "ks_pvalue": float(ks.pvalue),
            "params": str({
                "threshold_quantile": q,
                "threshold_u": u,
                "body_weight": w_body,
                "mu_log": mu,
                "sigma_log": sigma,
                "gpd_shape": xi,
                "gpd_scale": beta,
            }),
        }

        if (best is None) or (candidate["aic"] < best["aic"]):
            best = candidate

    if best is None:
        raise ValueError("No valid splice fit found.")

    return best


def fit_one_severity_candidate_set(x: np.ndarray) -> pd.DataFrame:
    fitters = [
        fit_lognormal_dist,
        fit_gamma_distn,
        fit_weibull_distn,
        fit_log_skew_normal_dist,
        fit_burr12_dist,
        fit_spliced_lognormal_gpd_dist,
    ]

    rows = []
    for fitter in fitters:
        try:
            rows.append(fitter(x))
        except Exception as e:
            rows.append({
                "model": fitter.__name__.replace("fit_", "").replace("_distn", "").replace("_dist", ""),
                "loglik": np.nan,
                "aic": np.nan,
                "bic": np.nan,
                "ks_stat": np.nan,
                "ks_pvalue": np.nan,
                "params": "FAILED: {0}".format(str(e)),
            })

    out = pd.DataFrame(rows)
    out = out.dropna(subset=["aic"]).sort_values(["aic", "bic", "ks_stat"]).reset_index(drop=True)
    return out


def fit_severity_distributions(
    sev_df: pd.DataFrame,
    cargo_col: str,
    amount_col: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    all_rows = []
    best_rows = []

    cargo_values = sorted(sev_df[cargo_col].dropna().unique())

    for cargo in cargo_values:
        x = sev_df.loc[sev_df[cargo_col] == cargo, amount_col].to_numpy(dtype=float)
        x = x[np.isfinite(x) & (x > 0)]

        if len(x) < 20:
            continue

        result = fit_one_severity_candidate_set(x)
        result.insert(0, "cargo_type", cargo)
        all_rows.append(result)

        best_rows.append(result.iloc[0].copy())

    all_results = pd.concat(all_rows, axis=0).reset_index(drop=True)
    best_results = pd.DataFrame(best_rows).reset_index(drop=True)

    return all_results, best_results


# ============================================================
# FREQUENCY GLMS
# ============================================================
def fit_frequency_glms(
    freq_df: pd.DataFrame,
    count_col: str,
    exposure_col: Optional[str],
) -> Tuple[pd.DataFrame, Any, pd.DataFrame]:
    numeric_candidates = [
        "cargo_value",
        "weight",
        "distance",
        "transit_duration",
        "pilot_experience",
        "vessel_age",
        "solar_radiation",
        "debris_density",
    ]

    categorical_candidates = [
        "route_risk",
        "cargo_type",
        "container_type",
    ]

    numeric_cols, categorical_cols = available_predictors(
        freq_df, numeric_candidates, categorical_candidates
    )

    formula = build_formula(count_col, numeric_cols, categorical_cols)

    train_df, test_df = train_test_split_df(freq_df, test_size=TEST_SIZE, seed=RANDOM_SEED)

    train_offset = None
    test_offset = None
    if exposure_col is not None and exposure_col in freq_df.columns:
        train_offset = np.log(pd.to_numeric(train_df[exposure_col], errors="coerce").fillna(0.0).clip(lower=1e-9))
        test_offset = np.log(pd.to_numeric(test_df[exposure_col], errors="coerce").fillna(0.0).clip(lower=1e-9))

    rows = []

    # Poisson GLM
    pois_model = smf.glm(
        formula=formula,
        data=train_df,
        family=sm.families.Poisson(),
        offset=train_offset,
    ).fit()

    mu_test = pois_model.predict(test_df, offset=test_offset)
    rows.append({
        "model": "Poisson GLM",
        "alpha": np.nan,
        "formula": formula,
        "train_aic": float(pois_model.aic),
        "train_bic": np.nan,
        "test_mean_deviance": mean_deviance(pois_model, test_df[count_col].to_numpy(), np.asarray(mu_test)),
        "test_rmse": rmse(test_df[count_col].to_numpy(), np.asarray(mu_test)),
    })

    best_model = pois_model
    best_score = rows[-1]["test_mean_deviance"]

    # NB GLM over alpha grid
    for alpha in NB_ALPHA_GRID:
        try:
            nb_model = smf.glm(
                formula=formula,
                data=train_df,
                family=sm.families.NegativeBinomial(alpha=alpha),
                offset=train_offset,
            ).fit()

            mu_test = nb_model.predict(test_df, offset=test_offset)
            score = mean_deviance(nb_model, test_df[count_col].to_numpy(), np.asarray(mu_test))

            rows.append({
                "model": "Negative Binomial GLM",
                "alpha": alpha,
                "formula": formula,
                "train_aic": float(nb_model.aic),
                "train_bic": np.nan,
                "test_mean_deviance": score,
                "test_rmse": rmse(test_df[count_col].to_numpy(), np.asarray(mu_test)),
            })

            if (np.isfinite(score)) and (score < best_score):
                best_score = score
                best_model = nb_model

        except Exception:
            continue

    results = pd.DataFrame(rows).sort_values(["test_mean_deviance", "train_aic"]).reset_index(drop=True)
    pvalues = extract_pvalue_table(best_model)

    return results, best_model, pvalues


# ============================================================
# SEVERITY GLMS
# ============================================================
def fit_severity_glms(
    sev_df: pd.DataFrame,
    amount_col: str,
) -> Tuple[pd.DataFrame, Any, pd.DataFrame]:
    sev_work = sev_df.copy()
    sev_work = sev_work[sev_work[amount_col] > 0].copy()

    numeric_candidates = [
        "cargo_value",
        "weight",
        "distance",
        "transit_duration",
        "pilot_experience",
        "vessel_age",
        "solar_radiation",
        "debris_density",
        "claim_seq",
    ]

    categorical_candidates = [
        "route_risk",
        "cargo_type",
        "container_type",
    ]

    numeric_cols, categorical_cols = available_predictors(
        sev_work, numeric_candidates, categorical_candidates
    )

    formula = build_formula(amount_col, numeric_cols, categorical_cols)

    train_df, test_df = train_test_split_df(sev_work, test_size=TEST_SIZE, seed=RANDOM_SEED)

    rows = []

    # Gamma log-link
    gamma_model = smf.glm(
        formula=formula,
        data=train_df,
        family=sm.families.Gamma(sm.families.links.log()),
    ).fit()

    mu_test = gamma_model.predict(test_df)
    rows.append({
        "model": "Gamma GLM",
        "link": "log",
        "formula": formula,
        "train_aic": float(gamma_model.aic),
        "test_mean_deviance": mean_deviance(gamma_model, test_df[amount_col].to_numpy(), np.asarray(mu_test)),
        "test_rmse": rmse(test_df[amount_col].to_numpy(), np.asarray(mu_test)),
    })

    best_model = gamma_model
    best_score = rows[-1]["test_mean_deviance"]

    # Inverse Gaussian log-link
    try:
        ig_model = smf.glm(
            formula=formula,
            data=train_df,
            family=sm.families.InverseGaussian(sm.families.links.log()),
        ).fit()

        mu_test = ig_model.predict(test_df)
        score = mean_deviance(ig_model, test_df[amount_col].to_numpy(), np.asarray(mu_test))

        rows.append({
            "model": "Inverse Gaussian GLM",
            "link": "log",
            "formula": formula,
            "train_aic": float(ig_model.aic),
            "test_mean_deviance": score,
            "test_rmse": rmse(test_df[amount_col].to_numpy(), np.asarray(mu_test)),
        })

        if (np.isfinite(score)) and (score < best_score):
            best_score = score
            best_model = ig_model

    except Exception:
        pass

    results = pd.DataFrame(rows).sort_values(["test_mean_deviance", "train_aic"]).reset_index(drop=True)
    pvalues = extract_pvalue_table(best_model)

    return results, best_model, pvalues


# ============================================================
# RUN
# ============================================================
freq_df, sev_df, meta = load_clean_data(CLEANED_FILE)

print("\nDetected columns")
print(pd.Series(meta))

# ----------------------------
# 1. Best frequency distribution
# ----------------------------
freq_dist_results, best_freq_dist = fit_frequency_distributions(
    freq_df=freq_df,
    count_col=meta["freq_count_col"],
)

print("\n=== FREQUENCY DISTRIBUTION CANDIDATES ===")
print(freq_dist_results.round(6))
print("\nBest frequency distribution:")
print(best_freq_dist)

# ----------------------------
# 2. Best severity distribution by cargo type
# ----------------------------
sev_dist_all, sev_dist_best = fit_severity_distributions(
    sev_df=sev_df,
    cargo_col=meta["sev_cargo_col"],
    amount_col=meta["sev_amount_col"],
)

print("\n=== SEVERITY DISTRIBUTION CANDIDATES (ALL) ===")
print(sev_dist_all.round(6))

print("\n=== BEST SEVERITY DISTRIBUTION BY CARGO TYPE ===")
print(sev_dist_best.round(6))


QQ Plot of Models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import lognorm, skewnorm, exponweib

# ============================================================
# CONFIG
# ============================================================
CLEANED_FILE = Path(
    "/Users/ben/Desktop/Uni/ACTL/ACTL4001/Assignment Code/Basic Data Cleaning and Modeling/srcsc-2026-claims-cargo_cleaned.xlsx"
)

SEV_SHEET = "sev"
OUTPUT_DIR = Path("qq_plots_severity_only")
OUTPUT_DIR.mkdir(exist_ok=True)

USE_LOG_AXES_FOR_SEVERITY = True

# ============================================================
# FITTED SEVERITY DISTRIBUTIONS
# ============================================================
SEVERITY_SPECS = {
    "cobalt": {
        "dist": "lognormal",
        "mu": 13.061442337216045,
        "sigma": 0.8569081381822045,
    },
    "gold": {
        "dist": "log_skew_normal",
        "shape": -0.7985638313815884,
        "loc": 17.68816928480843,
        "scale": 0.987915167223077,
    },
    "lithium": {
        "dist": "lognormal",
        "mu": 13.226583575398298,
        "sigma": 0.8816801497124209,
    },
    "platinum": {
        "dist": "log_skew_normal",
        "shape": -0.7358336912820225,
        "loc": 16.24010114678375,
        "scale": 0.9828604406948547,
    },
    "rare earths": {
        "dist": "log_skew_normal",
        "shape": 1.1745797426685858,
        "loc": 11.565385848921597,
        "scale": 1.050387375627892,
    },
    "supplies": {
        "dist": "log_exp_weibull",
        "a": 0.5032574573,
        "c": 2.0603365462,
        "loc_log": 10.3422251856,
        "scale_log": 1.3075827421,
    },
    "titanium": {
        "dist": "log_exp_weibull",
        "a": 0.5305097331,
        "c": 2.4126125838,
        "loc_log": 10.3422162683,
        "scale_log": 1.7126950865,
    },
}

# ============================================================
# HELPERS
# ============================================================
def find_first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def normalise_cargo_label(x):
    x = str(x).strip().lower()
    replacements = {
        "rare earth": "rare earths",
        "rare-earth": "rare earths",
        "rare-earths": "rare earths",
        "rare_earth": "rare earths",
        "rare_earths": "rare earths",
    }
    return replacements.get(x, x)


def get_plotting_positions(n):
    return (np.arange(1, n + 1) - 0.5) / n


def severity_theoretical_quantiles(probs, spec):
    dist = spec["dist"]

    if dist == "lognormal":
        # X ~ Lognormal(mu, sigma)
        return lognorm.ppf(
            probs,
            s=spec["sigma"],
            scale=np.exp(spec["mu"])
        )

    elif dist == "log_skew_normal":
        # log(X) ~ SkewNormal(shape, loc, scale)
        return np.exp(
            skewnorm.ppf(
                probs,
                a=spec["shape"],
                loc=spec["loc"],
                scale=spec["scale"],
            )
        )

    elif dist == "log_exp_weibull":
        # log(X) ~ Exponentiated Weibull(a, c, loc_log, scale_log)
        return np.exp(
            exponweib.ppf(
                probs,
                a=spec["a"],
                c=spec["c"],
                loc=spec["loc_log"],
                scale=spec["scale_log"],
            )
        )

    else:
        raise ValueError(f"Unknown distribution: {dist}")


def qq_plot(ax, theoretical, observed, title, log_axes=False):
    theoretical = np.asarray(theoretical, dtype=float)
    observed = np.asarray(observed, dtype=float)

    mask = np.isfinite(theoretical) & np.isfinite(observed)
    theoretical = theoretical[mask]
    observed = observed[mask]

    ax.scatter(theoretical, observed, s=18, alpha=0.7)

    lo = min(np.min(theoretical), np.min(observed))
    hi = max(np.max(theoretical), np.max(observed))
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1.5)

    ax.set_title(title)
    ax.set_xlabel("Theoretical quantiles")
    ax.set_ylabel("Empirical quantiles")
    ax.grid(True, alpha=0.3)

    if log_axes and np.all(theoretical > 0) and np.all(observed > 0):
        ax.set_xscale("log")
        ax.set_yscale("log")


# ============================================================
# LOAD SEVERITY DATA
# ============================================================
sev_df = pd.read_excel(CLEANED_FILE, sheet_name=SEV_SHEET)
sev_df = sev_df.loc[:, ~sev_df.columns.duplicated()].copy()

cargo_candidates = ["cargo_type", "cargo", "cargo_category", "type_of_cargo"]
sev_candidates = ["claim_amount", "severity", "claim_size", "loss_amount", "paid_amount"]

sev_cargo_col = find_first_existing_column(sev_df, cargo_candidates)
sev_amount_col = find_first_existing_column(sev_df, sev_candidates)

if sev_cargo_col is None:
    raise ValueError(
        f"Could not find cargo column. Columns found: {list(sev_df.columns)}"
    )

if sev_amount_col is None:
    raise ValueError(
        f"Could not find severity amount column. Columns found: {list(sev_df.columns)}"
    )

sev_df[sev_cargo_col] = sev_df[sev_cargo_col].astype(str).map(normalise_cargo_label)
sev_df[sev_amount_col] = pd.to_numeric(sev_df[sev_amount_col], errors="coerce")

sev_df = sev_df.dropna(subset=[sev_cargo_col, sev_amount_col]).copy()
sev_df = sev_df[sev_df[sev_amount_col] > 0].copy()

print("Using cargo column:", sev_cargo_col)
print("Using severity column:", sev_amount_col)
print("Unique cargo types in data:", sorted(sev_df[sev_cargo_col].dropna().unique()))

# ============================================================
# INDIVIDUAL QQ PLOTS
# ============================================================
for cargo, spec in SEVERITY_SPECS.items():
    x = sev_df.loc[sev_df[sev_cargo_col] == cargo, sev_amount_col].to_numpy(dtype=float)
    x = np.sort(x[np.isfinite(x) & (x > 0)])

    if len(x) == 0:
        print(f"Skipping {cargo}: no positive observations found.")
        continue

    p = get_plotting_positions(len(x))
    theoretical = severity_theoretical_quantiles(p, spec)

    fig, ax = plt.subplots(figsize=(7, 5))
    qq_plot(
        ax=ax,
        theoretical=theoretical,
        observed=x,
        title=f"Severity QQ Plot: {cargo.title()} vs {spec['dist']}",
        log_axes=USE_LOG_AXES_FOR_SEVERITY,
    )
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / f"qq_severity_{cargo.replace(' ', '_')}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

# ============================================================
# COMBINED PANEL
# ============================================================
cargo_list = list(SEVERITY_SPECS.keys())
fig, axes = plt.subplots(4, 2, figsize=(12, 20))
axes = axes.flatten()

for i, cargo in enumerate(cargo_list):
    ax = axes[i]
    spec = SEVERITY_SPECS[cargo]

    x = sev_df.loc[sev_df[sev_cargo_col] == cargo, sev_amount_col].to_numpy(dtype=float)
    x = np.sort(x[np.isfinite(x) & (x > 0)])

    if len(x) == 0:
        ax.set_title(f"{cargo.title()} (no data)")
        ax.axis("off")
        continue

    p = get_plotting_positions(len(x))
    theoretical = severity_theoretical_quantiles(p, spec)

    qq_plot(
        ax=ax,
        theoretical=theoretical,
        observed=x,
        title=cargo.title(),
        log_axes=USE_LOG_AXES_FOR_SEVERITY,
    )

for j in range(len(cargo_list), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "qq_severity_all_cargo_types.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print(f"\nAll severity QQ plots saved to: {OUTPUT_DIR.resolve()}")

Selected model parameter estimations

In [ ]:
# ============================================================
# SETTINGS
# ============================================================
BOOTSTRAP_ITERATIONS = 500
RANDOM_SEED = 42

# Assumes you already ran:
# freq_df, sev_df, meta = load_clean_data(CLEANED_FILE)

# ============================================================
# NEGATIVE BINOMIAL NB2
# Var(Y) = mu + alpha * mu^2
# ============================================================
def nb2_neg_loglik(params, y):
    log_mu, log_alpha = params
    mu = np.exp(log_mu)
    alpha = np.exp(log_alpha)

    if (not np.isfinite(mu)) or (not np.isfinite(alpha)) or (mu <= 0) or (alpha <= 0):
        return np.inf

    r = 1.0 / alpha
    p = r / (r + mu)

    return -np.sum(nbinom.logpmf(y, n=r, p=p))


def fit_nb2_params(y):
    y = np.asarray(y, dtype=float)

    y_mean = float(np.mean(y))
    y_var = float(np.var(y, ddof=1))
    alpha_start = max((y_var - y_mean) / max(y_mean ** 2, 1e-12), 1e-8)

    res = minimize(
        nb2_neg_loglik,
        x0=np.array([np.log(max(y_mean, 1e-8)), np.log(alpha_start)]),
        args=(y,),
        method="L-BFGS-B",
    )

    if not res.success:
        raise RuntimeError("NB fit failed")

    return {
        "mu": float(np.exp(res.x[0])),
        "alpha": float(np.exp(res.x[1])),
    }


# ============================================================
# SEVERITY PARAMETER FITS
# ============================================================
def fit_lognormal_params(x):
    x = np.asarray(x, dtype=float)
    logx = np.log(x)

    mu = float(np.mean(logx))
    sigma = float(np.std(logx, ddof=0))

    if sigma <= 0:
        raise RuntimeError("Invalid sigma")

    return {
        "mu": mu,
        "sigma": sigma,
    }


def fit_log_skew_normal_params(x):
    x = np.asarray(x, dtype=float)
    logx = np.log(x)

    shape, loc, scale = skewnorm.fit(logx)

    shape = float(shape)
    loc = float(loc)
    scale = float(scale)

    if scale <= 0:
        raise RuntimeError("Invalid scale")

    return {
        "shape": shape,
        "loc": loc,
        "scale": scale,
    }


def fit_log_exp_weibull_params(x):
    x = np.asarray(x, dtype=float)
    logx = np.log(x)

    a, c, loc, scale = exponweib.fit(logx)

    a = float(a)
    c = float(c)
    loc = float(loc)
    scale = float(scale)

    if scale <= 0:
        raise RuntimeError("Invalid scale")

    return {
        "a": a,
        "c": c,
        "loc_log": loc,
        "scale_log": scale,
    }


def fit_best_severity_model(x, model_name):
    if model_name == "lognormal":
        return fit_lognormal_params(x)
    elif model_name == "log_skew_normal":
        return fit_log_skew_normal_params(x)
    elif model_name == "log_exp_weibull":
        return fit_log_exp_weibull_params(x)
    else:
        raise ValueError(f"Unsupported model: {model_name}")


# ============================================================
# BOOTSTRAP FUNCTION
# ============================================================
def bootstrap_parameter_se(fit_func, data, n_boot=500, seed=42, min_success=100):
    rng = np.random.default_rng(seed)
    data = np.asarray(data)
    n = len(data)

    estimates = []

    for _ in range(n_boot):
        sample = rng.choice(data, size=n, replace=True)

        try:
            params = fit_func(sample)
            estimates.append(params)
        except Exception:
            continue

    if len(estimates) < min_success:
        raise RuntimeError(f"Too few successful bootstrap fits: {len(estimates)}")

    boot_df = pd.DataFrame(estimates)

    summary_rows = []
    for col in boot_df.columns:
        summary_rows.append({
            "parameter": col,
            "bootstrap_mean": boot_df[col].mean(),
            "std_error": boot_df[col].std(ddof=1),
            "ci_2.5%": boot_df[col].quantile(0.025),
            "ci_25%": boot_df[col].quantile(0.25),
            "ci_75%": boot_df[col].quantile(0.75),
            "ci_97.5%": boot_df[col].quantile(0.975),
            "n_success": len(boot_df),
        })

    summary_df = pd.DataFrame(summary_rows)
    return summary_df, boot_df


# ============================================================
# BEST MODELS FROM YOUR UPDATED TABLE
# ============================================================
BEST_SEV_MODELS = {
    "cobalt": "lognormal",
    "gold": "log_skew_normal",
    "lithium": "lognormal",
    "platinum": "log_skew_normal",
    "rare earths": "log_skew_normal",
    "supplies": "log_exp_weibull",
    "titanium": "log_exp_weibull",
}


# ============================================================
# 1) FREQUENCY PARAMETER STANDARD ERRORS
# ============================================================
freq_count_col = meta["freq_count_col"]
y_freq = freq_df[freq_count_col].to_numpy()

freq_point = fit_nb2_params(y_freq)

freq_se_summary, freq_boot = bootstrap_parameter_se(
    fit_func=fit_nb2_params,
    data=y_freq,
    n_boot=BOOTSTRAP_ITERATIONS,
    seed=RANDOM_SEED,
)

freq_point_df = pd.DataFrame({
    "parameter": list(freq_point.keys()),
    "point_estimate": list(freq_point.values()),
})

freq_final = freq_point_df.merge(freq_se_summary, on="parameter", how="left")

print("\n=== FREQUENCY PARAMETER STANDARD ERRORS ===")
print(freq_final.round(6))


# ============================================================
# 2) SEVERITY PARAMETER STANDARD ERRORS
# ============================================================
sev_cargo_col = meta["sev_cargo_col"]
sev_amount_col = meta["sev_amount_col"]

all_sev_results = []

for cargo, model_name in BEST_SEV_MODELS.items():
    x = sev_df.loc[sev_df[sev_cargo_col] == cargo, sev_amount_col].to_numpy(dtype=float)
    x = x[np.isfinite(x) & (x > 0)]

    if len(x) < 20:
        print(f"Skipping {cargo}: too few observations")
        continue

    point_est = fit_best_severity_model(x, model_name)

    se_summary, boot_df = bootstrap_parameter_se(
        fit_func=lambda sample, m=model_name: fit_best_severity_model(sample, m),
        data=x,
        n_boot=BOOTSTRAP_ITERATIONS,
        seed=RANDOM_SEED,
    )

    point_df = pd.DataFrame({
        "cargo_type": cargo,
        "distribution": model_name,
        "parameter": list(point_est.keys()),
        "point_estimate": list(point_est.values()),
    })

    merged = point_df.merge(se_summary, on="parameter", how="left")
    all_sev_results.append(merged)

sev_final = pd.concat(all_sev_results, ignore_index=True)

print("\n=== SEVERITY PARAMETER STANDARD ERRORS ===")
print(sev_final.round(6))


# ============================================================
# OPTIONAL: SAVE RESULTS
# ============================================================
output_path = CLEANED_FILE.parent / "parameter_standard_errors.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    freq_final.to_excel(writer, sheet_name="freq_param_se", index=False)
    sev_final.to_excel(writer, sheet_name="sev_param_se", index=False)

print("\nSaved to:", output_path)